# 04 – Custom Preprocessing (English Anime/Game Forum Corpus)

Pipeline: case folding & cleaning → tokenization → stopword removal → English Snowball stemming. Fungsi pipeline berada di `preprocessing.py` Hasil korpus disimpan terpisah sebagai `data/processed/korpus_preprocessed.json`.


In [1]:
%pip install pandas nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json
import pandas as pd


## 1. Fungsi dan library preprocessing

Fungsi utama disimpan di `shared_preprocessing.py` agar anggota 2, 3, dan 4 menggunakan implementasi yang sama. Notebook ini mengimpor fungsi tersebut untuk demonstrasi, pengujian, dan pemrosesan korpus. Stopword umum menggunakan daftar English dari NLTK, lalu ditambah stopword khusus elemen antarmuka forum.


In [3]:
import sys
from pathlib import Path

# Mendukung notebook yang dijalankan dari root proyek maupun folder preprocessing.
_candidates = [
    Path.cwd() / "preprocessing",
    Path.cwd(),
    Path.cwd().parent / "preprocessing",
]
MODULE_DIR = next(
    (path for path in _candidates if (path / "preprocessing.py").exists()),
    None,
)
if MODULE_DIR is None:
    raise FileNotFoundError(
        "shared_preprocessing.py tidak ditemukan. Pastikan folder preprocessing "
        "berada di root proyek."
    )

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from preprocessing import (
    ENGLISH_STOPWORDS,
    CUSTOM_STOPWORDS,
    STOPWORDS,
    clean_text,
    tokenize,
    remove_stopwords,
    stem_tokens,
    preprocess_text,
    preprocess_document,
)


## 2. Menentukan lokasi dan memuat korpus

In [4]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
INPUT_PATH = PROJECT_ROOT / "data/processed/korpus_clean.json"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/korpus_preprocessed.json"
print("Project:", PROJECT_ROOT)
print("Input tersedia:", INPUT_PATH.exists())

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    corpus = json.load(f)
print("Jumlah dokumen:", len(corpus))
pd.DataFrame(corpus[:3])[["source", "title", "url"]]

Project: d:\PrakViskom\UAS_PRAK_PI\kelompok4_ForumDiskusiAnimeGame\kelompok4_ForumDiskusiAnimeGame
Input tersedia: True
Jumlah dokumen: 4090


,source,title,url
0,MAL,Upcoming Dubbed Anime - Forums - MyAnimeList.net,https://myanimelist.net/forum/?topicid=1692966
1,MAL,Upcoming Dubbed Anime (4750 - ) - Forums - MyA...,https://myanimelist.net/forum/?topicid=1692966...
2,MAL,The 'Help Identifying This Anime/Character' Th...,https://myanimelist.net/forum/?topicid=1883466


## 3. Case folding dan cleaning

In [5]:
sample = "<p>I REALLY love this Anime! Visit https://example.com 😊</p>"
print("Sebelum:", sample)
print("Sesudah:", clean_text(sample))

Sebelum: <p>I REALLY love this Anime! Visit https://example.com 😊</p>
Sesudah: i really love this anime visit


## 4. Tokenization

In [6]:
sample = "The players are discussing anime episodes and RPG games."
print(tokenize(sample))

['the', 'players', 'are', 'discussing', 'anime', 'episodes', 'and', 'rpg', 'games']


## 5. Stopword removal

In [7]:
tokens = tokenize("This is not a good anime game, but the RPG has a bug")
print("Token awal       :", tokens)
print("Jumlah stopword library:", len(ENGLISH_STOPWORDS))
print("Custom stopwords :", sorted(CUSTOM_STOPWORDS))
print("Setelah stopword :", remove_stopwords(tokens))

Token awal       : ['this', 'is', 'not', 'a', 'good', 'anime', 'game', 'but', 'the', 'rpg', 'has', 'a', 'bug']
Jumlah stopword library: 175
Custom stopwords : ['avatar', 'click', 'edit', 'joined', 'likes', 'messages', 'offline', 'online', 'page', 'profile', 'quote', 'quoted', 'replies', 'reply', 'report', 'thread']
Setelah stopword : ['good', 'anime', 'game', 'rpg', 'bug']


Catatan: istilah domain seperti `anime`, `manga`, `game`, `RPG`, `NPC`, `PVP`, dan `DLC` tidak dimasukkan ke custom stopword karena dapat menjadi kata penting dalam pencarian. Stopword bahasa Inggris mengikuti daftar standar NLTK, sehingga kata seperti `no` dan `not` mengikuti daftar tersebut.


## 6. English stemming

In [8]:
tokens = remove_stopwords(tokenize("Players are playing games and discussing characters"))
print("Sebelum stemming:", tokens)
print("Sesudah stemming:", stem_tokens(tokens))

Sebelum stemming: ['players', 'playing', 'games', 'discussing', 'characters']
Sesudah stemming: ['player', 'play', 'game', 'discuss', 'charact']


In [9]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
snowball = SnowballStemmer("english")

In [10]:
words = [
    "anime",
    "manga",
    "character",
    "characters",
    "episode",
    "episodes",
    "playing",
    "players",
    "gameplay",
    "multiplayer",
    "discussing",
    "identifying",
    "upcoming",
    "studies",
    "better"
]

In [11]:
print(f"{'Original':<15} {'Porter':<15} {'Snowball':<15}")
print("-" * 45)

for word in words:
    print(
        f"{word:<15} "
        f"{porter.stem(word):<15} "
        f"{snowball.stem(word):<15}"
    )

Original        Porter          Snowball       
---------------------------------------------
anime           anim            anim           
manga           manga           manga          
character       charact         charact        
characters      charact         charact        
episode         episod          episod         
episodes        episod          episod         
playing         play            play           
players         player          player         
gameplay        gameplay        gameplay       
multiplayer     multiplay       multiplay      
discussing      discuss         discuss        
identifying     identifi        identifi       
upcoming        upcom           upcom          
studies         studi           studi          
better          better          better         


In [12]:
import json
from collections import Counter

with open(INPUT_PATH, "r", encoding="utf-8") as file:
    documents = json.load(file)

print("Jumlah dokumen:", len(documents))
print("Contoh dokumen:")
print(documents[0])

Jumlah dokumen: 4090
Contoh dokumen:
{'source': 'MAL', 'title': 'Upcoming Dubbed Anime - Forums - MyAnimeList.net', 'content': 'Dec 1, 2017 1:59 AM # 1 Kenny_Stryker Offline Mar 2017 1213 I\'ve tried searching for this thread, but couldn\'t find a similar thread, so I decided to make my own thread to keep a track of upcoming dubbed anime. \n\nSo, if you prefer dub anime, look no further than this. \n\nLast Updated: November 23, 2026 \n\n Currently Streaming SimulDubbed Anime - 45 \n Monday \n PSYREN (Episodes: 5/?) \n Tuesday \n One Piece (Episodes: 1160/????) \n The Classroom of a Black Cat and a Witch (Episodes: 1/24) \n The Laid-Off Cheat-Granting Mage Enjoys a Second Lease on Life (Episodes: 7/12) \n Wednesday \n A Returner\'s Magic Should Be Special Season 2 (Episodes: 7/12) \n Ingoku Danchi: Deviant\'s Apartment Complex (Episodes: 2/12) \n KAMUI: He\'s Behind You! (Episodes: 10/12) \n Magic Knight Rayearth (Episodes: 5/?) \n Nia Liston: The Merciless Maiden (Episodes: 1/25) \n Re

## 7. Uji pipeline lengkap

In [13]:
sample = "I really LOVE these Anime episodes! The players are discussing game bugs."
print("Original :", sample)
print("Processed:", preprocess_text(sample))

Original : I really LOVE these Anime episodes! The players are discussing game bugs.
Processed: realli love anim episod player discuss game bug


## 8. Contoh hasil pada dokumen korpus

In [14]:
sample_doc = corpus[0]
processed_sample = preprocess_document(sample_doc)
print("TITLE ASLI :", sample_doc.get("title", "")[:300])
print("TITLE HASIL:", processed_sample["title_processed"][:300])
print("CONTENT ASLI :", sample_doc.get("content", "")[:500])
print("CONTENT HASIL:", processed_sample["content_processed"][:500])

TITLE ASLI : Upcoming Dubbed Anime - Forums - MyAnimeList.net
TITLE HASIL: upcom dub anim forum myanimelist net
CONTENT ASLI : Dec 1, 2017 1:59 AM # 1 Kenny_Stryker Offline Mar 2017 1213 I've tried searching for this thread, but couldn't find a similar thread, so I decided to make my own thread to keep a track of upcoming dubbed anime. 

So, if you prefer dub anime, look no further than this. 

Last Updated: November 23, 2026 

 Currently Streaming SimulDubbed Anime - 45 
 Monday 
 PSYREN (Episodes: 5/?) 
 Tuesday 
 One Piece (Episodes: 1160/????) 
 The Classroom of a Black Cat and a Witch (Episodes: 1/24) 
 The Laid-Of
CONTENT HASIL: dec 2017 59 kenni stryker mar 2017 1213 i'v tri search find similar decid make keep track upcom dub anim prefer dub anim look last updat novemb 23 2026 current stream simuldub anim 45 monday psyren episod tuesday one piec episod 1160 classroom black cat witch episod 24 laid cheat grant mage enjoy second leas life episod 12 wednesday return magic special s

## 9. Memproses seluruh korpus dan menyimpan hasil

In [15]:
processed_corpus = [preprocess_document(doc) for doc in corpus]
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(processed_corpus, f, ensure_ascii=False, indent=2)
print("Jumlah dokumen diproses:", len(processed_corpus))
print("File output:", OUTPUT_PATH)
print("Field dokumen:", list(processed_corpus[0].keys()))

Jumlah dokumen diproses: 4090
File output: d:\PrakViskom\UAS_PRAK_PI\kelompok4_ForumDiskusiAnimeGame\kelompok4_ForumDiskusiAnimeGame\data\processed\korpus_preprocessed.json
Field dokumen: ['source', 'title', 'content', 'url', 'crawl_time', 'content_hash', 'title_processed', 'content_processed', 'text_processed']


## 10. Validasi hasil

In [16]:
assert len(processed_corpus) == len(corpus)
assert all("url" in doc and "text_processed" in doc for doc in processed_corpus)
assert all(isinstance(doc["text_processed"], str) for doc in processed_corpus)
print("Validasi berhasil: jumlah dokumen dan metadata penting tetap tersedia.")
pd.DataFrame([{
    "title": d.get("title", "")[:80],
    "text_processed": d.get("text_processed", "")[:120],
    "source": d.get("source", "")
} for d in processed_corpus[:5]])

Validasi berhasil: jumlah dokumen dan metadata penting tetap tersedia.


,title,text_processed,source
0,Upcoming Dubbed Anime - Forums - MyAnimeList.net,upcom dub anim forum myanimelist net dec 2017 ...,MAL
1,Upcoming Dubbed Anime (4750 - ) - Forums - MyA...,upcom dub anim 4750 forum myanimelist net oct ...,MAL
2,The 'Help Identifying This Anime/Character' Th...,help identifi anim charact v10 forum myanimeli...,MAL
3,The 'Help Identifying This Anime/Character' Th...,help identifi anim charact v10 7950 forum myan...,MAL
4,Do you think Anya and Saiki are LONG LOST RELA...,think anya saiki long lost relat forum myanime...,MAL
